In [9]:
#importlibraries

#libraries to extract frame
import os
import pandas as pd
import cv2
import numpy as np
import shutil

#libraries to undergo differencing
import torch
from torchvision import transforms
from PIL import Image
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt
from torchvision.utils import save_image
from sklearn.decomposition import PCA

#create a function that transforms images to tensors
convert_tensor = transforms.ToTensor()
convert_image = transforms.ToPILImage()

#set directories
#Imgae_to_video_locator_csv
image_to_vid_file = '../../data/Deepfish/location_data/location_dat.csv'
#image_to_vid_file = '../../data/Kakadu_fish/location_data/location_dat.csv'
#image_to_vid_file = '../../data/Tassy_BRUV/location_data/location_dat.csv'
#training data location
#folder with images, labels and yaml file are assumed to be within training data folder
training_dat_loc = '../../data/Deepfish/training_data/'
#training_dat_loc = '../../data/Kakadu_fish/training_data/'
#training_dat_loc = '../../data/Tassy_BRUV/training_data/'

#image size (if resizing needed for surrounding images)
#image_size = [1920,1080]
image_size = False
#if you do not want to resize surrounding images replace with False

#pixel location of time_stamp in video if wanted to be removed
#this blacks out top left corner of video to 635 pixels across and 70 down
#if you do not want to remove timestamp replace with False
#time_stamp_loc = [0,70,0,635] 
time_stamp_loc = False 

#whether to read in center frame from surrounding video/images
center_image_surround = True

#directory where differenced images will be saved
diff_folder = '../../data/Deepfish/augmented_images/'
#diff_folder = '../../data/Tassy_BRUV/augmented_images/'

#differenced image folder name
diff_name = 'test'

#differenced parameters
diff = 1
alpha = 10
method = "range"
red_layer = True
do_pca = True
normalise =False
log_im=False

In [10]:
#images
image_loc = training_dat_loc + 'images/'
#labels
label_loc = training_dat_loc + 'labels/'
#data.yaml directory
yaml_loc =  training_dat_loc + 'data.yaml'
#differencing location
diff_loc = diff_folder + diff_name + '/'


#prepare differenced directory
os.mkdir(diff_loc)
os.mkdir(diff_loc + '/images')
os.mkdir(diff_loc + '/images/train')
os.mkdir(diff_loc + '/images/test')
os.mkdir(diff_loc + '/images/valid')

#shutil.copytree(label_loc,diff_loc + '/labels')
shutil.copy(yaml_loc,diff_loc + '/data.yaml')

#import image location data
val_frame_dat = pd.read_csv(image_to_vid_file)

In [24]:
im_num = 10

#get image folder
img_folder = os.listdir(val_frame_dat['image_folder'][im_num])

#get index in image folder
im_index = img_folder.index(val_frame_dat['image_name'][im_num])

#extract left and right image
left_image_name = img_folder[im_index-1]
left_image = Image.open(val_frame_dat['image_folder'][im_num] + left_image_name)
right_image_name = img_folder[im_index+1]
right_image = Image.open(val_frame_dat['image_folder'][im_num] + right_image_name)

#give conditions if left or right image does not exist
if(left_image_name[:-7]!=val_frame_dat['image_name'][im_num][:-7]):
    left_image = right_image
if(right_image_name[:-7]!=val_frame_dat['image_name'][im_num][:-7]):
    right_image = left_image


#read in center image
if(center_image_surround==True):
    img_center = Image.open(val_frame_dat['image_location'][im_num])
else:
    img_center = Image.open(image_loc + val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])

#convert images to tensors
img_right_tens = convert_tensor(right_image)
img_left_tens = convert_tensor(left_image)
img_center_tens = convert_tensor(img_center)

#fix the left & right image before differencing
if(center_image_surround==True):
    img_center_tens = torch.stack((img_center_tens[2,:,:],img_center_tens[1,:,:],img_center_tens[0,:,:]),dim=0)
    if(image_size!=False):    
        img_center_tens = convert_tensor(convert_image(img_center_tens).resize(image_size))

img_right_tens_fixed = torch.stack((img_right_tens[2,:,:],img_right_tens[1,:,:],img_right_tens[0,:,:]),dim=0)
if(image_size!=False):    
    img_right_tens_fixed = convert_tensor(convert_image(img_right_tens_fixed).resize(image_size))
img_left_tens_fixed = torch.stack((img_left_tens[2,:,:],img_left_tens[1,:,:],img_left_tens[0,:,:]),dim=0)
if(image_size!=False):     
    img_left_tens_fixed = convert_tensor(convert_image(img_left_tens_fixed).resize(image_size))

#apply three frame differencing
if(method=="abs"):
    diff_im = alpha*(abs(img_center_tens-img_right_tens_fixed) + abs(img_center_tens-img_left_tens_fixed))
elif(method=="var"):
    avg_im = (img_center_tens+img_right_tens_fixed+img_left_tens_fixed)/3
    diff_im_unscaled = ((img_center_tens-avg_im)**2 + (img_right_tens_fixed-avg_im)**2+ (img_left_tens_fixed-avg_im)**2)/2
    diff_im = diff_im_unscaled*(1/torch.max(diff_im_unscaled))*alpha
elif(method=="range"):
    red_i = torch.stack((img_center_tens[0,:,:],img_right_tens_fixed[0,:,:],img_left_tens_fixed[0,:,:]),dim=0)
    green_i = torch.stack((img_center_tens[1,:,:],img_right_tens_fixed[1,:,:],img_left_tens_fixed[1,:,:]),dim=0)
    blue_i = torch.stack((img_center_tens[2,:,:],img_right_tens_fixed[2,:,:],img_left_tens_fixed[2,:,:]),dim=0)

    red_r = torch.max(red_i,dim=0)[0] - torch.min(red_i,dim=0)[0]
    green_r = torch.max(green_i,dim=0)[0] - torch.min(green_i,dim=0)[0]
    blue_r = torch.max(blue_i,dim=0)[0] - torch.min(blue_i,dim=0)[0]

    diff_im = torch.stack((red_r,green_r,blue_r),dim=0)*alpha
elif(method=="naive"):
    diff_im = alpha*(2*img_center_tens-img_right_tens_fixed-img_left_tens_fixed)  + 0.5
else:
    print("Please choose one of 'abs', 'var', 'range' or 'naive' for method.")
    #break
    
if(normalise==True):
    diff_mean = diff_im.mean()
    diff_std = diff_im.std()
    diff_im = (diff_im - diff_mean)/diff_std
else:
    pass
    
if(log_im==True):
    diff_im = np.log(diff_im+1)
else:
    pass

if(red_layer==True):
    diff_im_avg = (diff_im[0,:,:]+diff_im[1,:,:]+diff_im[2,:,:])/3
    if(do_pca==True):
        # Get the image shape
        im_depth, im_height, im_width,  = img_center_tens.shape

        # Reshape the image tensor into a 2D matrix
        im_matrix = np.reshape(convert_image(img_center_tens), (im_height * im_width, im_depth))

        # Perform PCA on the matrix
        pca = PCA(n_components=2,whiten=True)
        reduced_matrix = pca.fit_transform(im_matrix)

        # Reshape the reduced matrix back into a 3D tensor
        reduced_image = convert_tensor(np.reshape(reduced_matrix, (im_height,im_width, 2)))

        #apply sigmoid transform
        reduced_image = torch.sigmoid(reduced_image)
        
        diff_im_final = torch.stack((diff_im_avg,reduced_image[0,:,:],reduced_image[1,:,:]),dim=0)
    else:
        diff_im_final = torch.stack((diff_im_avg,img_center_tens[1,:,:],img_center_tens[2,:,:]),dim=0)
else:
    diff_im_final=diff_im

if(time_stamp_loc!=False):
    diff_im_final[:,time_stamp_loc[0]:time_stamp_loc[1],time_stamp_loc[2]:time_stamp_loc[3]]  = 0    

        
    #save image
save_image(diff_im_final,diff_loc +"/images/"+ val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])
#convert_image(diff_im_final)

In [25]:
im_num_list = list(range(0,val_frame_dat.shape[0]))

for im_num in im_num_list:
    #get image folder
    img_folder = os.listdir(val_frame_dat['image_folder'][im_num])

    #get index in image folder
    im_index = img_folder.index(val_frame_dat['image_name'][im_num])

    #extract left and right image
    left_image_name = img_folder[im_index-1]
    left_image = Image.open(val_frame_dat['image_folder'][im_num] + left_image_name)
    right_image_name = img_folder[im_index+1]
    right_image = Image.open(val_frame_dat['image_folder'][im_num] + right_image_name)

    #give conditions if left or right image does not exist
    if(left_image_name[:-7]!=val_frame_dat['image_name'][im_num][:-7]):
        left_image = right_image
    if(right_image_name[:-7]!=val_frame_dat['image_name'][im_num][:-7]):
        right_image = left_image


    #read in center image
    if(center_image_surround==True):
        img_center = Image.open(val_frame_dat['image_location'][im_num])
    else:
        img_center = Image.open(image_loc + val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])

    #convert images to tensors
    img_right_tens = convert_tensor(right_image)
    img_left_tens = convert_tensor(left_image)
    img_center_tens = convert_tensor(img_center)

    #fix the left & right image before differencing
    if(center_image_surround==True):
        img_center_tens = torch.stack((img_center_tens[2,:,:],img_center_tens[1,:,:],img_center_tens[0,:,:]),dim=0)
        if(image_size!=False):    
            img_center_tens = convert_tensor(convert_image(img_center_tens).resize(image_size))

    img_right_tens_fixed = torch.stack((img_right_tens[2,:,:],img_right_tens[1,:,:],img_right_tens[0,:,:]),dim=0)
    if(image_size!=False):    
        img_right_tens_fixed = convert_tensor(convert_image(img_right_tens_fixed).resize(image_size))
    img_left_tens_fixed = torch.stack((img_left_tens[2,:,:],img_left_tens[1,:,:],img_left_tens[0,:,:]),dim=0)
    if(image_size!=False):     
        img_left_tens_fixed = convert_tensor(convert_image(img_left_tens_fixed).resize(image_size))

    #apply three frame differencing
    if(method=="abs"):
        diff_im = alpha*(abs(img_center_tens-img_right_tens_fixed) + abs(img_center_tens-img_left_tens_fixed))
    elif(method=="var"):
        avg_im = (img_center_tens+img_right_tens_fixed+img_left_tens_fixed)/3
        diff_im_unscaled = ((img_center_tens-avg_im)**2 + (img_right_tens_fixed-avg_im)**2+ (img_left_tens_fixed-avg_im)**2)/2
        diff_im = diff_im_unscaled*(1/torch.max(diff_im_unscaled))*alpha
    elif(method=="range"):
        red_i = torch.stack((img_center_tens[0,:,:],img_right_tens_fixed[0,:,:],img_left_tens_fixed[0,:,:]),dim=0)
        green_i = torch.stack((img_center_tens[1,:,:],img_right_tens_fixed[1,:,:],img_left_tens_fixed[1,:,:]),dim=0)
        blue_i = torch.stack((img_center_tens[2,:,:],img_right_tens_fixed[2,:,:],img_left_tens_fixed[2,:,:]),dim=0)

        red_r = torch.max(red_i,dim=0)[0] - torch.min(red_i,dim=0)[0]
        green_r = torch.max(green_i,dim=0)[0] - torch.min(green_i,dim=0)[0]
        blue_r = torch.max(blue_i,dim=0)[0] - torch.min(blue_i,dim=0)[0]

        diff_im = torch.stack((red_r,green_r,blue_r),dim=0)*alpha
    elif(method=="naive"):
        diff_im = alpha*(2*img_center_tens-img_right_tens_fixed-img_left_tens_fixed)  + 0.5
    else:
        print("Please choose one of 'abs', 'var', 'range' or 'naive' for method.")
        #break
        
    if(normalise==True):
        diff_mean = diff_im.mean()
        diff_std = diff_im.std()
        diff_im = (diff_im - diff_mean)/diff_std
    else:
        pass
        
    if(log_im==True):
        diff_im = np.log(diff_im+1)
    else:
        pass

    if(red_layer==True):
        diff_im_avg = (diff_im[0,:,:]+diff_im[1,:,:]+diff_im[2,:,:])/3
        if(do_pca==True):
            # Get the image shape
            im_depth, im_height, im_width,  = img_center_tens.shape

            # Reshape the image tensor into a 2D matrix
            im_matrix = np.reshape(convert_image(img_center_tens), (im_height * im_width, im_depth))

            # Perform PCA on the matrix
            pca = PCA(n_components=2,whiten=True)
            reduced_matrix = pca.fit_transform(im_matrix)

            # Reshape the reduced matrix back into a 3D tensor
            reduced_image = convert_tensor(np.reshape(reduced_matrix, (im_height,im_width, 2)))

            #apply sigmoid transform
            reduced_image = torch.sigmoid(reduced_image)
            
            diff_im_final = torch.stack((diff_im_avg,reduced_image[0,:,:],reduced_image[1,:,:]),dim=0)
        else:
            diff_im_final = torch.stack((diff_im_avg,img_center_tens[1,:,:],img_center_tens[2,:,:]),dim=0)
    else:
        diff_im_final=diff_im

    if(time_stamp_loc!=False):
        diff_im_final[:,time_stamp_loc[0]:time_stamp_loc[1],time_stamp_loc[2]:time_stamp_loc[3]]  = 0    

            
        #save image
    save_image(diff_im_final,diff_loc +"/images/"+ val_frame_dat['split'][im_num] + "/" + val_frame_dat['image_name'][im_num])
#convert_image(diff_im_final)

KeyboardInterrupt: 